# Hyperparameter Tuning

We will go through through the implementation of finding the best hyper parameters in a given neural network. Following are the hyper parameters which we are going to discuss:

   - Optimization algorithms
   - Activation function
   - Learning rate
   - Weights
   - Bias
   - Number of nodes in a layer
   - L0, L1 and L2 regularization
   - Number of epochs

For each of these, we will train the model for 100 epochs and provide the results in Tensor board for us to visualize. Once, we have the optimum value from each of these, then we will use them to improve our model to identify a Bollywood character's age group.

Most of the code in the above eight hyper parameters is same initially, which actually deals with loading necessary libraries, reading training data, converting it to a form suitable for the neural network, normalizing it and label encoding the output classes.

## Preprocessing

### Installing Packages

In [1]:
%pip install numpy pandas scikit-learn tensorflow keras imageio pillow

Note: you may need to restart the kernel to use updated packages.


### Importing libraries

In [2]:
import os
import re
import numpy as np
import pandas as pd
from keras.models import Sequential
from keras.layers import Dense, Flatten, InputLayer
from sklearn.preprocessing import LabelEncoder
from tensorflow.python.keras import utils
import keras
import imageio 
from PIL import Image
import tensorflow as tf

2025-06-15 12:28:18.645753: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-15 12:28:18.650547: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-15 12:28:18.659991: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749990498.675010   45876 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749990498.679543   45876 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1749990498.692398   45876 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

### Reading the data

In [3]:
import zipfile

# Reading the data from zipped CSV files
with zipfile.ZipFile('datasets/agedetectiontrain.zip') as z:
	with z.open('train.csv') as f:
		train = pd.read_csv(f)

### Image resizing of train data into single numpy array

In [4]:
temp = []
# Zipfile reading and image processing
with zipfile.ZipFile('datasets/agedetectiontrain.zip') as z:
    for img_name in train.ID:
        with z.open(f'Train/{img_name}') as img_file:
            img = imageio.imread(img_file)
            img = np.array(Image.fromarray(img).resize((32, 32))).astype('float32')
            temp.append(img)
train_x = np.stack(temp)

/tmp/ipykernel_45876/2118266606.py:6: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  img = imageio.imread(img_file)


### Normalizing the images

In [5]:
train_x = train_x / 255.

### Encoding the categorical variable to numeric

In [6]:
lb = LabelEncoder()
train_y = lb.fit_transform(train.Class)
train_y = keras.utils.to_categorical(train_y)

### Specifying all the parameters we will be using in our network

In [7]:
input_num_units = (32, 32, 3)
hidden_num_units = 500
output_num_units = 3
epochs = 100
batch_size = 128

## Optimization Algorithms

We are going to focus on the following five optimization algorithms:

   - SGD
   - Adagrad
   - Adadelta
   - RMSprop
   - Adam

### Defining the network

First, let us define the network, as shown below:

In [8]:
model = Sequential([
  InputLayer(input_shape=input_num_units),
  Flatten(),
  Dense(units=hidden_num_units, activation='relu'),
  Dense(units=output_num_units, activation='softmax'),
])

/home/codespace/.python/current/lib/python3.12/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(
2025-06-15 12:28:39.378313: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


### Building model functions

Next, let us write a function to build a model for each optimizer and save the accuracy, loss, validation accuracy and validation loss for Tensor board visualization.

In [9]:
def models_with_different_optimizers(list_of_optimizers):    
    
    for i in range(len(list_of_optimizers)):        
        model.compile(loss='categorical_crossentropy',
                  optimizer=list_of_optimizers[i], # Learning rate and momentum can be passed inside optimizer
                  metrics=['accuracy'])
        # Traning the model and writing log files for TensorBoard in distinct directories        
        val = re.search('optimizers\..*\so', str(list_of_optimizers[i])).group(0)[11:][:-2] # Fetching optimizer name
        logdir = f'optims/{val}' # Each log file needs to be written in a distinct directory. (Mandatory)
        
        # Writing graph will take time. Hence, keeping it False.
        cb = keras.callbacks.TensorBoard(log_dir=logdir, write_graph=False)         
        print('Building model using '+ val + ' optimizer')
        history = model.fit(train_x, train_y, epochs=epochs, 
                           validation_split=0.2,
                           callbacks=[cb])
        print('Model built sucessfully.')
        print('')

<>:8: SyntaxWarning: invalid escape sequence '\.'
<>:8: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_45876/1457440352.py:8: SyntaxWarning: invalid escape sequence '\.'
  val = re.search('optimizers\..*\so', str(list_of_optimizers[i])).group(0)[11:][:-2] # Fetching optimizer name


### Listing the optimizers

In [10]:
optims = [keras.optimizers.Adam(), keras.optimizers.Adadelta(), 
          keras.optimizers.Adagrad(), keras.optimizers.RMSprop(), 
          keras.optimizers.SGD()]

### Calling the function

In [11]:
models_with_different_optimizers(optims)

2025-06-15 12:28:39.512904: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 195674112 exceeds 10% of free system memory.


Building model using adam.Adam optimizer
Epoch 1/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.5537 - loss: 1.0313 - val_accuracy: 0.6258 - val_loss: 0.8171
Epoch 2/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.6055 - loss: 0.8359 - val_accuracy: 0.6135 - val_loss: 0.8329
Epoch 3/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.6177 - loss: 0.8226 - val_accuracy: 0.6266 - val_loss: 0.7951
Epoch 4/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.6283 - loss: 0.8094 - val_accuracy: 0.6447 - val_loss: 0.7852
Epoch 5/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.6489 - loss: 0.7855 - val_accuracy: 0.6457 - val_loss: 0.7782
Epoch 6/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.6566 - loss: 0.7691 - val_accuracy: 0.6522 - val_loss: 0.7703
Epoch 7/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.6495 - loss: 0.7727 - val_accuracy: 0.6457 - val_loss: 0.7847
Epoch 8/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/

2025-06-15 12:37:21.821446: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 195674112 exceeds 10% of free system memory.


498/498 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.7972 - loss: 0.4785 - val_accuracy: 0.6888 - val_loss: 0.8567
Epoch 2/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.8007 - loss: 0.4676 - val_accuracy: 0.6904 - val_loss: 0.8488
Epoch 3/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.8069 - loss: 0.4574 - val_accuracy: 0.6929 - val_loss: 0.8430
Epoch 4/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.8084 - loss: 0.4569 - val_accuracy: 0.6956 - val_loss: 0.8388
Epoch 5/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.8189 - loss: 0.4343 - val_accuracy: 0.6956 - val_loss: 0.8356
Epoch 6/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.8203 - loss: 0.4364 - val_accuracy: 0.6976 - val_loss: 0.8332
Epoch 7/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - accuracy: 0.8199 - loss: 0.4307 - val_accuracy: 0.6994 - val_loss: 0.8312
Epoch 8/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - accuracy: 0.8189 - loss: 0.4359 - val_accura

2025-06-15 12:48:47.538055: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 195674112 exceeds 10% of free system memory.


498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.8269 - loss: 0.4213 - val_accuracy: 0.7029 - val_loss: 0.8240
Epoch 2/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.8306 - loss: 0.4166 - val_accuracy: 0.7017 - val_loss: 0.8248
Epoch 3/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.8410 - loss: 0.3982 - val_accuracy: 0.7057 - val_loss: 0.8251
Epoch 4/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.8328 - loss: 0.4135 - val_accuracy: 0.7024 - val_loss: 0.8261
Epoch 5/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.8363 - loss: 0.4129 - val_accuracy: 0.7042 - val_loss: 0.8263
Epoch 6/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.8393 - loss: 0.4006 - val_accuracy: 0.7034 - val_loss: 0.8260
Epoch 7/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.8400 - loss: 0.4022 - val_accuracy: 0.7037 - val_loss: 0.8271
Epoch 8/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.8350 - loss: 0.4089 - val_accuracy: 0.7024

2025-06-15 12:56:02.750977: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 195674112 exceeds 10% of free system memory.


498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.8004 - loss: 0.4804 - val_accuracy: 0.6999 - val_loss: 0.8814
Epoch 2/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.8039 - loss: 0.4654 - val_accuracy: 0.6818 - val_loss: 0.8922
Epoch 3/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.8023 - loss: 0.4673 - val_accuracy: 0.6544 - val_loss: 1.0066
Epoch 4/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.7952 - loss: 0.4760 - val_accuracy: 0.6605 - val_loss: 0.9048
Epoch 5/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.7995 - loss: 0.4784 - val_accuracy: 0.6954 - val_loss: 0.8804
Epoch 6/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 0.7987 - loss: 0.4724 - val_accuracy: 0.7104 - val_loss: 0.8828
Epoch 7/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.7937 - loss: 0.4845 - val_accuracy: 0.6891 - val_loss: 0.9539
Epoch 8/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.8017 - loss: 0.4774 - val_accuracy: 0.

2025-06-15 13:03:59.102190: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 195674112 exceeds 10% of free system memory.


498/498 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.8672 - loss: 0.3300 - val_accuracy: 0.6921 - val_loss: 1.2426
Epoch 2/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8731 - loss: 0.3117 - val_accuracy: 0.6914 - val_loss: 1.2582
Epoch 3/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.8737 - loss: 0.3075 - val_accuracy: 0.6806 - val_loss: 1.2570
Epoch 4/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8754 - loss: 0.3105 - val_accuracy: 0.6891 - val_loss: 1.2692
Epoch 5/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8754 - loss: 0.3008 - val_accuracy: 0.6833 - val_loss: 1.2863
Epoch 6/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8769 - loss: 0.3081 - val_accuracy: 0.6894 - val_loss: 1.2912
Epoch 7/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8733 - loss: 0.3099 - val_accuracy: 0.6883 - val_loss: 1.2934
Epoch 8/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.8754 - loss: 0.3061 - val_accuracy: 0.6896

This will result in a folder named “optims” which consists of the evaluation metrics. We can visualize the output by running the following command:

In [3]:
!tensorboard --logdir optims

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard
2025-06-15 13:53:57.480521: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-15 13:53:57.484002: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-15 13:53:57.494374: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749995637.509753    4730 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749995637.514115    4730 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1749995637.528624    4730 computation_p